# 6. Solar Data Integration PipelineEnd-to-end pipeline that combines NREL, NASA, Berkeley, and Kaggle data into a master feature table.**Owner:** _assign team member_**Status:** In Progress

# Pipeline


site input = one property/system you care about

NREL = modeled solar production for that site

NASA = weather/solar conditions for that site

Berkeley / Tracking the Sun = local market benchmarks from real installations

Kaggle supplement = optional extra summary features

master_df = one combined row with all of that information

In [ ]:
# =========================
# SOLAR DATA INTEGRATION PIPELINE
# =========================

# Optional installs for Colab / notebooks
# !pip install -q gdown requests pandas numpy python-dotenv kagglehub

import os
import zipfile
import requests
import numpy as np
import pandas as pd

# -------------------------
# 1) CONFIG
# -------------------------

# --- Load API key (works in both Colab and local) ---
try:
    from google.colab import userdata
    nrel = userdata.get('NREL_API_KEY')
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=os.path.join('..', '.env'))
    nrel = os.environ.get('NREL_API_KEY')

assert nrel, "NREL API key not found. Add to .env or Colab Secrets."

# Project site / property inputs
SITE_INPUT = {
    "site_id": "demo_site_001",
    "address_label": "San Jose, CA",
    "lat": 37.33,
    "lon": -121.8863,
    "system_capacity_kw": 5.0,
    "azimuth": 180,
    "tilt": 20,
    "array_type": 1,
    "module_type": 0,
    "losses": 14,
    "state": "CA",
    "zip_code": 95192,
    "customer_segment": "RES",
}

# Tracking the Sun download info
FILE_ID = "1NQh4TRC_IqDz2r5vfZuxDm6LGjEuexdu"
ZIP_OUTPUT = "tracking_the_sun.zip"
TTS_CSV = "TTS_LBNL_public_file_29-Sep-2025_all.csv"

# Kaggle supplemental file path
import kagglehub
kaggle_path = kagglehub.dataset_download('shaistashahid/urban-solar-roi-and-sustainability')
KAGGLE_SUPP_PATH = os.path.join(kaggle_path, 'solar_energy_worldwide.csv')

# -------------------------
# 2) HELPERS
# -------------------------
def safe_get(d, *keys, default=np.nan):
    """
    Safely walk nested JSON dictionaries/lists.

    Why? --> API reponses are nested dictionaries (JSON format).
    So we are making a helper method to parse the API response data structure
    this prevents crashes, missing data, or improper formating/organization
    """
    cur = d
    try:
        for k in keys:
            if isinstance(cur, list):
                cur = cur[k]
            else:
                cur = cur.get(k)
        return cur if cur is not None else default
    except Exception:
        return default

def fetch_pvwatts(site, api_key):
    """
    Query NREL PVWatts v8 and return a flat dict.

    This function calls the NREL PVWatts API.
          input: latitude
                 longitude
                 system_capacity_kw
                 azimuth.  ---> describes compass direction that solar array is facing/pointed (typically toward equator) impacts efficiency percentage
                 tilt --> vertical slope
                 array_type
                 module_type
                 losses
          output: predicted annual AC electricity generation (kwh)
                  annual solar radiation
                  how effectively the system produces relative to max possible
                  monthly means for POA, DC, AC
                  weather station metadata
    """
    url = "https://developer.nrel.gov/api/pvwatts/v8.json"
    params = {
        "api_key": api_key,
        "lat": site["lat"],
        "lon": site["lon"],
        "system_capacity": site["system_capacity_kw"],
        "azimuth": site["azimuth"],
        "tilt": site["tilt"],
        "array_type": site["array_type"],
        "module_type": site["module_type"],
        "losses": site["losses"],
    }

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    j = r.json()

    outputs = j.get("outputs", {})
    station_info = j.get("station_info", {})

    row = {
        "site_id": site["site_id"],
        "pvwatts_ac_annual_kwh": outputs.get("ac_annual", np.nan),
        "pvwatts_solrad_annual": outputs.get("solrad_annual", np.nan),
        "pvwatts_capacity_factor": outputs.get("capacity_factor", np.nan),
        "pvwatts_poa_monthly_mean": np.nan if "poa_monthly" not in outputs else np.nanmean(outputs["poa_monthly"]),
        "pvwatts_dc_monthly_mean": np.nan if "dc_monthly" not in outputs else np.nanmean(outputs["dc_monthly"]),
        "pvwatts_ac_monthly_mean": np.nan if "ac_monthly" not in outputs else np.nanmean(outputs["ac_monthly"]),
        "pvwatts_station_distance_m": station_info.get("distance", np.nan),
        "pvwatts_station_lat": station_info.get("lat", np.nan),
        "pvwatts_station_lon": station_info.get("lon", np.nan),
        "pvwatts_version": j.get("version", np.nan),
    }
    return pd.DataFrame([row])

def fetch_nasa_power(site, start="20240101", end="20240131"):
    """
    Query NASA POWER hourly point data and aggregate to simple summary features.

    taking inputs
        # ALLSKY... = global horizontal irradiance / sunlight at surface
        # includes temperature, relative humidity, and wind speed.
    returns
        global horizontal irradiance: power per unit area received from Sun (EMF w/in sensor range).
          mean, average midday, and max
        temp, humidity, wind speed

    """
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
    params = {
        # ALLSKY... = global horizontal irradiance / sunlight at surface
        # includes temperature, relative humidity, and wind speed.
        "parameters": "ALLSKY_SFC_SW_DWN,T2M,RH2M,WS2M",
        "community": "SB",
        "longitude": site["lon"],
        "latitude": site["lat"],
        "start": start,
        "end": end,
        "format": "JSON"
    }

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    j = r.json()

    param_block = safe_get(j, "properties", "parameter", default={})

    # Build hourly frame from available variables
    hourly_frames = []
    for var_name, series in param_block.items():
        s = pd.Series(series, name=var_name)
        s.index = pd.to_datetime(s.index, format="%Y%m%d%H")
        hourly_frames.append(s)

    if not hourly_frames:
        return pd.DataFrame([{"site_id": site["site_id"]}])

    hourly = pd.concat(hourly_frames, axis=1).reset_index().rename(columns={"index": "datetime"})
    hourly["hour"] = hourly["datetime"].dt.hour
    hourly["month"] = hourly["datetime"].dt.month

    # Flatten to summarized features
    out = {
        "site_id": site["site_id"],
        "nasa_hours": len(hourly),
        "nasa_ghi_mean": hourly["ALLSKY_SFC_SW_DWN"].mean() if "ALLSKY_SFC_SW_DWN" in hourly else np.nan,
        "nasa_ghi_max": hourly["ALLSKY_SFC_SW_DWN"].max() if "ALLSKY_SFC_SW_DWN" in hourly else np.nan,
        "nasa_temp_mean_c": hourly["T2M"].mean() if "T2M" in hourly else np.nan,
        "nasa_temp_max_c": hourly["T2M"].max() if "T2M" in hourly else np.nan,
        "nasa_rh_mean": hourly["RH2M"].mean() if "RH2M" in hourly else np.nan,
        "nasa_ws_mean": hourly["WS2M"].mean() if "WS2M" in hourly else np.nan,
        "nasa_midday_ghi_mean": hourly.loc[hourly["hour"].between(10, 14), "ALLSKY_SFC_SW_DWN"].mean()
            if "ALLSKY_SFC_SW_DWN" in hourly else np.nan,
    }

    return pd.DataFrame([out])

def load_tracking_the_sun():
    """
    Download/unzip/load Berkeley: Tracking the Sun if not present.

    """
    if not os.path.exists(TTS_CSV):
        if not os.path.exists(ZIP_OUTPUT):
            import gdown
            url = f"https://drive.google.com/uc?id={FILE_ID}"
            gdown.download(url, ZIP_OUTPUT, quiet=False)

        with zipfile.ZipFile(ZIP_OUTPUT, "r") as z:
            z.extractall(".")

    df = pd.read_csv(TTS_CSV, encoding="latin1", low_memory=False)
    return df

def build_tts_benchmarks(tts_df, site):
    """
    Create benchmark features from Berkeley / Tracking the Sun.
    Tries ZIP-level first, then state-level fallback.

    Gets the local subset of data from config (San Jose) from larger (Berkeley) database
    returns subset near example given from configs.

    checks zip match, state match, and customer segment match.
    Runs merge
    """
    cols_needed = [
        "installation_date",
        "PV_system_size_DC",
        "total_installed_price",
        "customer_segment",
        "state",
        "zip_code",
        "tracking",
        "ground_mounted",
        "azimuth_1",
        "tilt_1",
        "efficiency_module_1",
        "battery_rated_capacity_kWh"
    ]
    existing = [c for c in cols_needed if c in tts_df.columns]
    df = tts_df[existing].copy()

    # Basic cleanup
    if "installation_date" in df.columns:
        df["installation_date"] = pd.to_datetime(df["installation_date"], errors="coerce")

    for c in ["PV_system_size_DC", "total_installed_price", "tilt_1", "azimuth_1",
              "efficiency_module_1", "battery_rated_capacity_kWh"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if {"total_installed_price", "PV_system_size_DC"}.issubset(df.columns):
        df["price_per_watt"] = df["total_installed_price"] / (df["PV_system_size_DC"] * 1000)

    # Filters
    zip_match = pd.Series([True] * len(df))
    state_match = pd.Series([True] * len(df))
    segment_match = pd.Series([True] * len(df))

    if "zip_code" in df.columns and pd.notna(site.get("zip_code")):
        zip_match = df["zip_code"].astype(str) == str(site["zip_code"])

    if "state" in df.columns and pd.notna(site.get("state")):
        state_match = df["state"].astype(str) == str(site["state"])

    if "customer_segment" in df.columns and pd.notna(site.get("customer_segment")):
        segment_match = df["customer_segment"].astype(str) == str(site["customer_segment"])

    local_df = df[zip_match & segment_match].copy()
    if len(local_df) < 30:
        local_df = df[state_match & segment_match].copy()
        geo_level = "state"
    else:
        geo_level = "zip"

    result = {
        "site_id": site["site_id"],
        "tts_geo_level_used": geo_level,
        "tts_sample_size": len(local_df),
        "tts_median_system_size_dc": local_df["PV_system_size_DC"].median() if "PV_system_size_DC" in local_df else np.nan,
        "tts_median_price_per_watt": local_df["price_per_watt"].median() if "price_per_watt" in local_df else np.nan,
        "tts_median_tilt": local_df["tilt_1"].median() if "tilt_1" in local_df else np.nan,
        "tts_median_azimuth": local_df["azimuth_1"].median() if "azimuth_1" in local_df else np.nan,
        "tts_tracking_rate": local_df["tracking"].astype(str).str.lower().isin(["y", "yes", "true", "1"]).mean()
            if "tracking" in local_df else np.nan,
        "tts_ground_mount_rate": local_df["ground_mounted"].astype(str).str.lower().isin(["y", "yes", "true", "1"]).mean()
            if "ground_mounted" in local_df else np.nan,
        "tts_module_efficiency_median": local_df["efficiency_module_1"].median() if "efficiency_module_1" in local_df else np.nan,
        "tts_battery_kwh_median": local_df["battery_rated_capacity_kWh"].median() if "battery_rated_capacity_kWh" in local_df else np.nan,
    }

    # Optional recent installations only
    if "installation_date" in local_df.columns:
        recent = local_df[local_df["installation_date"] >= pd.Timestamp("2022-01-01")]
        result["tts_recent_sample_size"] = len(recent)
        result["tts_recent_median_price_per_watt"] = recent["price_per_watt"].median() if "price_per_watt" in recent else np.nan

    return pd.DataFrame([result])

def load_kaggle_supplement(path):
    """
    Load Kaggle supplemental dataset if present.
    Keeps only numeric summary features to avoid brittle joins.
    """
    if not os.path.exists(path):
        print(f"[INFO] Kaggle supplemental file not found: {path}")
        return pd.DataFrame([{"site_id": SITE_INPUT["site_id"]}])

    kdf = pd.read_csv(path, low_memory=False)

    numeric_cols = kdf.select_dtypes(include=[np.number]).columns.tolist()
    summary = {"site_id": SITE_INPUT["site_id"]}

    for col in numeric_cols[:25]:  # cap to keep feature set manageable
        summary[f"kaggle_mean_{col}"] = kdf[col].mean()
        summary[f"kaggle_median_{col}"] = kdf[col].median()

    summary["kaggle_rows"] = len(kdf)
    summary["kaggle_num_numeric_cols"] = len(numeric_cols)

    return pd.DataFrame([summary])

# -------------------------
# 3) RUN PIPELINE
#     site input: San Jose, CA
# -------------------------
site_df = pd.DataFrame([SITE_INPUT])

pvwatts_df = fetch_pvwatts(SITE_INPUT, nrel)
nasa_df = fetch_nasa_power(SITE_INPUT, start="20240101", end="20240131")

tts_raw = load_tracking_the_sun()
tts_bench_df = build_tts_benchmarks(tts_raw, SITE_INPUT)

kaggle_df = load_kaggle_supplement(KAGGLE_SUPP_PATH)

# Unified modeling / analysis table
master_df = (
    site_df
    .merge(pvwatts_df, on="site_id", how="left")
    .merge(nasa_df, on="site_id", how="left")
    .merge(tts_bench_df, on="site_id", how="left")
    .merge(kaggle_df, on="site_id", how="left")
)

# -------------------------
# 4) OPTIONAL DERIVED FEATURES
# -------------------------
if {"pvwatts_ac_annual_kwh", "system_capacity_kw"}.issubset(master_df.columns):
    master_df["annual_kwh_per_kw"] = master_df["pvwatts_ac_annual_kwh"] / master_df["system_capacity_kw"]

if {"tts_median_price_per_watt", "system_capacity_kw"}.issubset(master_df.columns):
    master_df["estimated_system_cost_from_tts"] = (
        master_df["tts_median_price_per_watt"] * master_df["system_capacity_kw"] * 1000
    )

if {"pvwatts_ac_annual_kwh", "estimated_system_cost_from_tts"}.issubset(master_df.columns):
    master_df["kwh_per_dollar_est"] = (
        master_df["pvwatts_ac_annual_kwh"] / master_df["estimated_system_cost_from_tts"]
    )

# -------------------------
# 5) DISPLAY / EXPORT
# -------------------------
print("MASTER FEATURE TABLE SHAPE:", master_df.shape)
display(master_df.T)

# Optional save
master_df.to_csv("solar_master_feature_table.csv", index=False)
print("\nSaved: solar_master_feature_table.csv")

In [ ]:
display(master_df.T)